# Laboratorium terbuka: model dua spesies

Notebook ini merupakan pendamping komputasi mandiri untuk Bab 6. Seluruh bidang fase dihitung dengan NumPy, SciPy, dan Matplotlib, dapat dijalankan secara luring, dan tidak memakai kode MATLAB, PPLANE, atau perangkat proprieter.

Notebook memeriksa model predator–mangsa Lotka–Volterra, model kompetisi, Soal 3, dan Soal 4. Parameter gambar adalah nilai demonstrasi yang dinyatakan di setiap bagian.

**ID unit:** O005-LEGA-V101-CH06  
**ID notebook:** O005-LEGA-V101-CH06-NB01  
**Lisensi notebook:** CC BY-NC-SA 4.0  
**Asal komponen:** pendamping komputasi baru berdasarkan persamaan dan soal Bab 6; bukan salinan kode aplikasi yang ditautkan sumber.

## 1. Medan, batas biologis, titik tetap, dan linearisasi

Model kanonis predator–mangsa adalah

$$f'=a f(1-bf-s),\qquad s'=s(f-1),$$

sedangkan model kompetisi adalah

$$x'=x(1-x-a y),\qquad y'=c y(1-y-bx).$$

Faktor $f,s,x,y$ membuat setiap sumbu koordinat invarian. Medan polinomial bersifat unik secara lokal, sehingga lintasan yang berawal nonnegatif tidak dapat melintasi sumbu menuju nilai negatif.

In [ ]:
import os
import platform

import numpy as np
import scipy
from scipy.integrate import solve_ivp
import matplotlib

if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.set_printoptions(precision=8, suppress=True)

def medan_lv(waktu, z, a, b):
    f, s = z
    return np.array([a * f * (1.0 - b * f - s), s * (f - 1.0)])

def jacobian_lv(f, s, a, b):
    return np.array([
        [a * (1.0 - 2.0 * b * f - s), -a * f],
        [s, f - 1.0],
    ])

def medan_kompetisi(waktu, z, a, b, c):
    x, y = z
    return np.array([x * (1.0 - x - a * y), c * y * (1.0 - y - b * x)])

def jacobian_kompetisi(x, y, a, b, c):
    return np.array([
        [1.0 - 2.0 * x - a * y, -a * x],
        [-b * c * y, c * (1.0 - 2.0 * y - b * x)],
    ])

# Kedua sumbu benar-benar invarian untuk dua model.
assert medan_lv(0.0, [0.0, 2.0], 1.0, 0.5)[0] == 0.0
assert medan_lv(0.0, [2.0, 0.0], 1.0, 0.5)[1] == 0.0
assert medan_kompetisi(0.0, [0.0, 2.0], 0.5, 0.8, 1.0)[0] == 0.0
assert medan_kompetisi(0.0, [2.0, 0.0], 0.5, 0.8, 1.0)[1] == 0.0

# Titik tetap dan spektrum Lotka–Volterra dengan redaman logistik b=0,5.
a_lv, b_lv = 1.0, 0.5
P0_lv = np.array([0.0, 0.0])
P1_lv = np.array([1.0 / b_lv, 0.0])
P2_lv = np.array([1.0, 1.0 - b_lv])
for P in (P0_lv, P1_lv, P2_lv):
    assert np.allclose(medan_lv(0.0, P, a_lv, b_lv), 0.0)
eig_P0_lv = np.linalg.eigvals(jacobian_lv(*P0_lv, a_lv, b_lv))
eig_P1_lv = np.linalg.eigvals(jacobian_lv(*P1_lv, a_lv, b_lv))
eig_P2_lv = np.linalg.eigvals(jacobian_lv(*P2_lv, a_lv, b_lv))
assert np.allclose(np.sort(eig_P0_lv), [-1.0, 1.0])
assert np.allclose(np.sort(eig_P1_lv), [-1.0, 1.0])
assert np.all(eig_P2_lv.real < 0.0) and np.any(np.abs(eig_P2_lv.imag) > 0.0)

# Pada b=0, titik (1,1) adalah pusat linear dengan nilai eigen ±i sqrt(a).
eig_pusat = np.linalg.eigvals(jacobian_lv(1.0, 1.0, a_lv, 0.0))
assert np.allclose(np.sort_complex(eig_pusat), np.sort_complex(np.array([-1j, 1j])))

print(f"Python/NumPy/SciPy/Matplotlib: {platform.python_version()} / {np.__version__} / {scipy.__version__} / {matplotlib.__version__}")
print("LV b=0,5, eigen P0/P1/P2:", eig_P0_lv, eig_P1_lv, eig_P2_lv)
print("LV b=0, eigen pusat:", eig_pusat)

In [ ]:
def gambar_medan(ax, fungsi, x_batas, y_batas, parameter, judul):
    x = np.linspace(x_batas[0], x_batas[1], 25)
    y = np.linspace(y_batas[0], y_batas[1], 25)
    X, Y = np.meshgrid(x, y)
    U = np.empty_like(X)
    V = np.empty_like(Y)
    for indeks in np.ndindex(X.shape):
        U[indeks], V[indeks] = fungsi(0.0, [X[indeks], Y[indeks]], *parameter)
    norma = np.hypot(U, V)
    norma[norma == 0.0] = 1.0
    ax.streamplot(X, Y, U / norma, V / norma, density=0.8, color="#BBBBBB", linewidth=0.7)
    ax.set(xlim=x_batas, ylim=y_batas, xlabel="mangsa / spesies x", ylabel="predator / spesies y", title=judul)
    ax.grid(alpha=0.18)

def integrasi_lv(awal, a, b, akhir):
    hasil = solve_ivp(
        medan_lv,
        (0.0, akhir),
        np.asarray(awal, dtype=float),
        args=(a, b),
        method="DOP853",
        rtol=1e-11,
        atol=1e-13,
        dense_output=False,
        max_step=0.08,
    )
    assert hasil.success
    assert np.min(hasil.y) > 0.0
    return hasil

awal_lv = ([0.35, 1.70], [2.00, 1.30], [1.70, 0.20])
hasil_redam = [integrasi_lv(z0, 1.0, 0.5, 40.0) for z0 in awal_lv]
galat_akhir_redam = max(np.linalg.norm(h.y[:, -1] - P2_lv) for h in hasil_redam)
assert galat_akhir_redam < 0.002

hasil_tutup = integrasi_lv([0.35, 1.50], 1.0, 0.0, 60.0)
f_tutup, s_tutup = hasil_tutup.y
invarian_H = f_tutup - np.log(f_tutup) + (s_tutup - np.log(s_tutup))
drift_H = float(np.max(np.abs(invarian_H - invarian_H[0])))
assert drift_H < 1e-8
assert np.linalg.norm(hasil_tutup.y[:, -1] - np.array([1.0, 1.0])) > 0.1

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11.4, 4.8), constrained_layout=True)
gambar_medan(ax0, medan_lv, (0.0, 3.0), (0.0, 2.4), (1.0, 0.5), "Predator–mangsa teredam: a=1, b=0,5")
for hasil in hasil_redam:
    ax0.plot(hasil.y[0], hasil.y[1], linewidth=1.6)
ax0.plot(*P2_lv, "o", color="#D55E00", label="$P_2=(1,0{,}5)$")
ax0.legend()

gambar_medan(ax1, medan_lv, (0.0, 3.0), (0.0, 2.4), (1.0, 0.0), "Lotka–Volterra klasik: a=1, b=0")
ax1.plot(f_tutup, s_tutup, color="#0072B2", linewidth=1.8)
ax1.plot(1.0, 1.0, "o", color="#D55E00", label="$P_2=(1,1)$")
ax1.legend()
plt.show()

print(f"Galat maksimum menuju P2 untuk b=0,5: {galat_akhir_redam:.3e}")
print(f"Drift maksimum invarian H untuk b=0: {drift_H:.3e}")

## 2. Koreksi aljabar model kompetisi

Daftar titik tetap yang benar adalah

$$P_0=(0,0),\quad P_1=(1,0),\quad P_2=(0,1),\quad P_3=\left(\frac{1-a}{1-ab},\frac{1-b}{1-ab}\right).$$

Sumber beku menggandakan $P_1$ ketika menulis $P_2$, meskipun analisis berikutnya memakai $(0,1)$. Sumber juga mencetak diskriminan yang tidak sama dengan $T^2-4D$. Bentuk yang benar adalah

$$\Delta=\frac{[a-1+c(b-1)]^2-4c(a-1)(b-1)(1-ab)}{(1-ab)^2}.$$

Sel berikut memeriksa kedua koreksi dan empat rezim ekologis: koeksistensi, eksklusi bistabil, serta dominasi salah satu dari dua spesies.

In [ ]:
def titik_tetap_kompetisi(a, b):
    dasar = [np.array([0.0, 0.0]), np.array([1.0, 0.0]), np.array([0.0, 1.0])]
    penyebut = 1.0 - a * b
    if np.isclose(penyebut, 0.0):
        return dasar + [None]
    return dasar + [np.array([(1.0 - a) / penyebut, (1.0 - b) / penyebut])]

def data_interior(a, b, c):
    P3 = titik_tetap_kompetisi(a, b)[3]
    assert P3 is not None
    J = jacobian_kompetisi(*P3, a, b, c)
    T = float(np.trace(J))
    D = float(np.linalg.det(J))
    delta_matriks = T * T - 4.0 * D
    delta_benar = (
        (a - 1.0 + c * (b - 1.0)) ** 2
        - 4.0 * c * (a - 1.0) * (b - 1.0) * (1.0 - a * b)
    ) / (1.0 - a * b) ** 2
    delta_tercetak = ((a - 1.0) - c * (b - 1.0)) ** 2 / (1.0 - a * b) ** 2
    assert np.isclose(delta_matriks, delta_benar)
    return P3, J, T, D, delta_benar, delta_tercetak

# Titik sumbu ketiga adalah (0,1), bukan pengulangan (1,0).
P_koreksi = titik_tetap_kompetisi(0.5, 0.8)
assert np.array_equal(P_koreksi[2], np.array([0.0, 1.0]))
for P in P_koreksi:
    assert np.allclose(medan_kompetisi(0.0, P, 0.5, 0.8, 1.0), 0.0)

# Koeksistensi stabil: a<1 dan b<1.
P3_ko, J_ko, T_ko, D_ko, delta_ko, delta_sumber = data_interior(0.5, 0.8, 1.0)
eig_ko = np.linalg.eigvals(J_ko)
assert np.all(P3_ko > 0.0) and np.all(eig_ko < 0.0)
assert not np.isclose(delta_ko, delta_sumber)

# Eksklusi bistabil: a>1 dan b>1; P3 adalah pelana, P1 dan P2 stabil.
P3_bi, J_bi, T_bi, D_bi, delta_bi, _ = data_interior(1.2, 2.0, 1.0)
eig_bi = np.linalg.eigvals(J_bi)
eig_bi_P1 = np.linalg.eigvals(jacobian_kompetisi(1.0, 0.0, 1.2, 2.0, 1.0))
eig_bi_P2 = np.linalg.eigvals(jacobian_kompetisi(0.0, 1.0, 1.2, 2.0, 1.0))
assert D_bi < 0.0 and eig_bi.min() < 0.0 < eig_bi.max()
assert np.all(eig_bi_P1 < 0.0) and np.all(eig_bi_P2 < 0.0)

# Soal 3: a<1<b; P3 berada di luar kuadran dan P1 adalah satu-satunya penarik biologis.
a_s3, b_s3, c_s3 = 0.6, 1.4, 1.2
P3_s3 = titik_tetap_kompetisi(a_s3, b_s3)[3]
eig_s3_P1 = np.linalg.eigvals(jacobian_kompetisi(1.0, 0.0, a_s3, b_s3, c_s3))
eig_s3_P2 = np.linalg.eigvals(jacobian_kompetisi(0.0, 1.0, a_s3, b_s3, c_s3))
assert np.any(P3_s3 < 0.0)
assert np.all(eig_s3_P1 < 0.0)
assert eig_s3_P2.min() < 0.0 < eig_s3_P2.max()

# Dominasi sebaliknya: a>1>b; P2 stabil dan P1 pelana.
eig_bal_P1 = np.linalg.eigvals(jacobian_kompetisi(1.0, 0.0, 1.4, 0.6, 1.2))
eig_bal_P2 = np.linalg.eigvals(jacobian_kompetisi(0.0, 1.0, 1.4, 0.6, 1.2))
assert eig_bal_P1.min() < 0.0 < eig_bal_P1.max()
assert np.all(eig_bal_P2 < 0.0)

print("Koeksistensi P3/eigen:", P3_ko, eig_ko)
print(f"Diskriminan benar/tercetak untuk (0,5;0,8;1): {delta_ko:.8f} / {delta_sumber:.8f}")
print("Eksklusi bistabil P3/eigen:", P3_bi, eig_bi)
print("Soal 3 P3 di luar kuadran; eigen P1/P2:", P3_s3, eig_s3_P1, eig_s3_P2)

In [ ]:
def integrasi_kompetisi(awal, a, b, c, akhir):
    hasil = solve_ivp(
        medan_kompetisi,
        (0.0, akhir),
        np.asarray(awal, dtype=float),
        args=(a, b, c),
        method="DOP853",
        rtol=1e-10,
        atol=1e-12,
        max_step=0.12,
    )
    assert hasil.success
    assert np.min(hasil.y) > 0.0
    return hasil

awal_kompetisi = ([0.20, 2.00], [2.00, 0.20], [0.35, 0.45], [2.20, 1.80])
kasus = (
    (0.5, 0.8, 1.0, 45.0, "Koeksistensi: a=0,5; b=0,8"),
    (a_s3, b_s3, c_s3, 80.0, "Soal 3, x dominan: a=0,6; b=1,4"),
    (1.2, 2.0, 1.0, 55.0, "Eksklusi bistabil: a=1,2; b=2"),
)

fig, sumbu = plt.subplots(1, 3, figsize=(15.2, 4.7), constrained_layout=True)
semua_hasil = []
for ax, (a, b, c, akhir, judul) in zip(sumbu, kasus):
    gambar_medan(ax, medan_kompetisi, (0.0, 2.5), (0.0, 2.5), (a, b, c), judul)
    hasil_kasus = [integrasi_kompetisi(z0, a, b, c, akhir) for z0 in awal_kompetisi]
    semua_hasil.append(hasil_kasus)
    for hasil in hasil_kasus:
        ax.plot(hasil.y[0], hasil.y[1], linewidth=1.5)
plt.show()

hasil_s3 = semua_hasil[1]
galat_s3 = np.array([np.linalg.norm(h.y[:, -1] - np.array([1.0, 0.0])) for h in hasil_s3])
assert np.max(galat_s3) < 0.002
assert all(h.y[1, -1] < h.y[1, 0] for h in hasil_s3)

akhir_ko = np.column_stack([h.y[:, -1] for h in semua_hasil[0]])
assert np.max(np.linalg.norm(akhir_ko - P3_ko[:, None], axis=0)) < 0.002

akhir_bi = np.column_stack([h.y[:, -1] for h in semua_hasil[2]])
jarak_ke_sumbu = np.minimum(
    np.linalg.norm(akhir_bi - np.array([[1.0], [0.0]]), axis=0),
    np.linalg.norm(akhir_bi - np.array([[0.0], [1.0]]), axis=0),
)
assert np.max(jarak_ke_sumbu) < 0.003

print("Keadaan akhir Soal 3 (kolom) =\n", np.column_stack([h.y[:, -1] for h in hasil_s3]))
print("Galat Soal 3 terhadap (1,0):", galat_s3)
print("Keadaan akhir kasus eksklusi bistabil (kolom) =\n", akhir_bi)

### Kesimpulan Soal 3

Untuk $a<1<b$, kandidat titik interior selalu mempunyai satu koordinat negatif (atau tidak ada ketika $ab=1$). Titik $(1,0)$ mempunyai nilai eigen $-1$ dan $c(1-b)<0$, sedangkan $(0,1)$ mempunyai nilai eigen $1-a>0$ dan $-c<0$. Karena solusi interior terbatas dan kriteria Dulac dengan pengali $1/(xy)$ memberi divergensi $-1/y-c/x<0$, tidak ada orbit periodik interior. Semua keadaan awal dengan $x_0,y_0>0$ menuju $(1,0)$: spesies $x$ mengeksklusi spesies $y$.

## 3. Soal 4: titik tetap, kestabilan, dan kelayakan biologis

Untuk

$$x'=4-x(1+y),\qquad y'=y(1+y-x),$$

nullcline adalah $x=4/(1+y)$ serta $y=0$ atau $x=1+y$. Dalam kuadran nonnegatif, perpotongannya adalah $(4,0)$ dan $(2,1)$.

Model menjaga nonnegativitas: pada $x=0$, $x'=4>0$, sedangkan pada $y=0$, $y'=0$. Akan tetapi, ia tidak menjamin keberadaan global. Jika $M=\max\{x_0,4\}$, maka $x(t)\le M$; untuk $y\ge2(M-1)$ berlaku $y'\ge y^2/2$, sehingga perbandingan dengan persamaan Riccati menunjukkan ledakan waktu hingga. Ini merupakan cacat model populasi global, bukan sekadar masalah skala grafik.

In [ ]:
def medan_soal4(waktu, z):
    x, y = z
    return np.array([4.0 - x - x * y, -x * y + y + y * y])

def jacobian_soal4(x, y):
    return np.array([[-1.0 - y, -x], [-y, -x + 1.0 + 2.0 * y]])

P4_stabil = np.array([4.0, 0.0])
P4_pelana = np.array([2.0, 1.0])
assert np.allclose(medan_soal4(0.0, P4_stabil), 0.0)
assert np.allclose(medan_soal4(0.0, P4_pelana), 0.0)
assert medan_soal4(0.0, [0.0, 3.0])[0] == 4.0
assert medan_soal4(0.0, [3.0, 0.0])[1] == 0.0

eig_4_stabil = np.sort(np.linalg.eigvals(jacobian_soal4(*P4_stabil)))
eig_4_pelana = np.sort(np.linalg.eigvals(jacobian_soal4(*P4_pelana)))
eig_4_pelana_eksak = np.sort(np.array([(-1.0 - np.sqrt(17.0)) / 2.0, (-1.0 + np.sqrt(17.0)) / 2.0]))
assert np.allclose(eig_4_stabil, [-3.0, -1.0])
assert np.allclose(eig_4_pelana, eig_4_pelana_eksak)
assert np.isclose(np.trace(jacobian_soal4(*P4_stabil)), -4.0) and np.isclose(np.linalg.det(jacobian_soal4(*P4_stabil)), 3.0)
assert np.isclose(np.trace(jacobian_soal4(*P4_pelana)), -1.0) and np.isclose(np.linalg.det(jacobian_soal4(*P4_pelana)), -4.0)

awal_stabil = ([3.5, 0.05], [4.5, 0.05], [4.0, 0.10])
hasil_stabil_4 = []
for z0 in awal_stabil:
    hasil = solve_ivp(medan_soal4, (0.0, 18.0), z0, method="DOP853", rtol=1e-11, atol=1e-13, max_step=0.05)
    assert hasil.success and np.min(hasil.y) >= 0.0
    hasil_stabil_4.append(hasil)
galat_stabil_4 = max(np.linalg.norm(h.y[:, -1] - P4_stabil) for h in hasil_stabil_4)
assert galat_stabil_4 < 1e-6

def ambang_y(waktu, z):
    return z[1] - 50.0

ambang_y.terminal = True
ambang_y.direction = 1
hasil_tumbuh_4 = solve_ivp(
    medan_soal4,
    (0.0, 1.0),
    [0.0, 8.0],
    events=ambang_y,
    method="DOP853",
    rtol=1e-11,
    atol=1e-13,
    max_step=0.002,
)
assert hasil_tumbuh_4.success and len(hasil_tumbuh_4.t_events[0]) == 1
waktu_ambang = float(hasil_tumbuh_4.t_events[0][0])
assert 0.0 < waktu_ambang < 0.25

fig, ax = plt.subplots(figsize=(7.6, 5.4), constrained_layout=True)
gambar_medan(ax, lambda t, z: medan_soal4(t, z), (0.0, 6.0), (0.0, 8.0), (), "Soal 4: simpul, pelana, dan pertumbuhan tak terbatas")
for hasil in hasil_stabil_4:
    ax.plot(hasil.y[0], hasil.y[1], linewidth=1.7)
ax.plot(hasil_tumbuh_4.y[0], hasil_tumbuh_4.y[1], color="#D55E00", linewidth=1.8, label="menuju ledakan")
ax.plot(*P4_stabil, "o", color="#009E73", label="simpul stabil (4,0)")
ax.plot(*P4_pelana, "s", color="#CC79A7", label="pelana (2,1)")
ax.legend()
plt.show()

print("Soal 4 eigen (4,0):", eig_4_stabil)
print("Soal 4 eigen (2,1):", eig_4_pelana)
print(f"Galat lintasan stabil terhadap (4,0): {galat_stabil_4:.3e}")
print(f"Lintasan (0,8) mencapai y=50 pada t={waktu_ambang:.8f}")

## 4. Soal 5: dimensi dan bentuk tak berdimensi

Biarkan $[X]=\mathsf X$, $[Y]=\mathsf Y$, dan $[t]=T$. Dari kesetaraan dimensi setiap suku diperoleh

$$[\alpha]=T^{-1},\quad[\beta]=\mathsf X T^{-1},\quad[\gamma]=\mathsf X^{-1}T^{-1},\quad[\delta]=T^{-1},\quad[\zeta]=\mathsf Y^{-1}T^{-1}.$$

Skala yang selalu positif adalah $X_0=\beta/\alpha$, $Y_0=\alpha/\zeta$, dan $\tau=\alpha t$. Dengan $x=X/X_0$ dan $y=Y/Y_0$,

$$\frac{dx}{d\tau}=1-x,\qquad \frac{dy}{d\tau}=y(\rho x-\mu-y),\quad \rho=\frac{\gamma\beta}{\alpha^2},\quad\mu=\frac\delta\alpha.$$

Pemilihan ini menormalkan keseimbangan pasokan–peluruhan $X$, skala waktu peluruhan $X$, dan koefisien pembatas kuadrat $Y$.

In [ ]:
alpha, beta, gamma, delta, zeta = 2.0, 6.0, 0.5, 0.4, 0.25
X0, Y0 = beta / alpha, alpha / zeta
rho, mu = gamma * beta / alpha**2, delta / alpha
X_uji, Y_uji = 1.7, 2.2
x_uji, y_uji = X_uji / X0, Y_uji / Y0

dX_dt = -alpha * X_uji + beta
dY_dt = gamma * X_uji * Y_uji - delta * Y_uji - zeta * Y_uji**2
dx_dtau_dari_dimensi = dX_dt / (alpha * X0)
dy_dtau_dari_dimensi = dY_dt / (alpha * Y0)
dx_dtau_kanonis = 1.0 - x_uji
dy_dtau_kanonis = y_uji * (rho * x_uji - mu - y_uji)

assert X0 > 0.0 and Y0 > 0.0
assert rho > 0.0 and mu > 0.0
assert np.isclose(dx_dtau_dari_dimensi, dx_dtau_kanonis)
assert np.isclose(dy_dtau_dari_dimensi, dy_dtau_kanonis)

print(f"Skala X0={X0:.8f}, Y0={Y0:.8f}; parameter rho={rho:.8f}, mu={mu:.8f}")
print(f"Pemeriksaan ruas kanan tak berdimensi: dx/dtau={dx_dtau_kanonis:.8f}, dy/dtau={dy_dtau_kanonis:.8f}")

## 5. Bukti primer dan reduksi Soal 6

Fussmann dkk., *Science* **290** (2000), hlm. 1358, Persamaan (1)–(4), mengidentifikasi predator sebagai rotifera *Brachionus calyciflorus* dan mangsa sebagai alga *Chlorella vulgaris*. Notasinya adalah $N$ untuk konsentrasi nitrogen dinamis yang membatasi pertumbuhan alga, $C$ untuk Chlorella, $R$ untuk Brachionus reproduktif, dan $B$ untuk total Brachionus.

Reduksi tanpa pemisahan demografis memerlukan penutupan baru $R=B$. Di bawah asumsi ini, persamaan $R$ dan parameter transisi $\lambda$ dihapus, persamaan $N$ dan $C$ dipertahankan, dan persamaan predator menjadi

$$\frac{dB}{dt}=F_B(C)B-(\delta+m)B.$$

Persamaan tiga variabel $(N,C,B)$ tersebut adalah penyederhanaan tambahan untuk menjawab soal, bukan klaim bahwa $R=B$ merupakan identitas dalam model empat persamaan artikel.

## Kesimpulan

Sel komputasi membuktikan invariansi sumbu, memeriksa titik tetap dan nilai eigen, membandingkan orbit Lotka–Volterra teredam dengan orbit tertutup, mengukur drift invarian, mengoreksi titik tetap serta diskriminan model kompetisi, menjalankan bidang fase Soal 3, dan mengaudit kestabilan serta ledakan waktu hingga pada Soal 4. Semua lintasan berasal dari integrasi ODE terbuka dengan masukan tetap dan toleransi eksplisit.